# 97. nomic-embed-text-v1.5 総合評価

## 目的
- nomic-ai/nomic-embed-text-v1.5 (Dense, ~100Mパラメータ, 768D) を多角的に評価
- NB96 (v2-moe) との比較: 軽量Denseモデルがどこまで迫れるか
- 特にCPU推論速度に注目（100M vs v2-moeの305M active）

## モデル特性
| 項目 | v1.5 | v2-moe (NB96) |
|------|------|----------------|
| アーキテクチャ | Dense | MoE (8 experts, top-2) |
| パラメータ | ~100M | 475M total / 305M active |
| 次元 | 768 (Matryoshka 64-768) | 768 (Matryoshka 256-768) |
| 最大トークン | 8192 | 512 |
| 言語 | **英語主体** | ~100言語 |
| プレフィックス | search_document: / search_query: | 同左 |

## 注意
- v1.5は英語主体モデルのため、JA評価は参考値（性能低下が予想される）

## 0. Setup

In [1]:
import numpy as np
import time
import gc
import torch
import sys
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from scipy.stats import spearmanr, pearsonr

sys.path.insert(0, '../src')
from itq_lsh import ITQLSH

DATA_DIR = Path('../data')
np.random.seed(42)

MODEL_ID = 'nomic-ai/nomic-embed-text-v1.5'
MODEL_KEY = 'nomic_v1_5'
N_SAMPLES = 10000
MAX_CHARS = 500

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


## 1. データ準備 (Wikipedia JA/EN 10K)

In [2]:
from datasets import load_dataset

def collect_wikipedia(lang_code: str, n_samples: int = N_SAMPLES) -> list[str]:
    """Wikipedia からテキストを収集（プレフィックスなし）"""
    wiki = load_dataset(
        'wikimedia/wikipedia',
        f'20231101.{lang_code}',
        split='train',
        streaming=True
    )
    documents = []
    for i, item in enumerate(tqdm(wiki, total=n_samples, desc=f'Wikipedia {lang_code}')):
        if i >= n_samples:
            break
        text = item['text'][:MAX_CHARS].strip()
        if len(text) < 50:
            continue
        documents.append(text)
    print(f'Collected {len(documents):,} documents ({lang_code})')
    return documents

print('=== Japanese ===')
docs_ja = collect_wikipedia('ja')
print(f'\n=== English ===')
docs_en = collect_wikipedia('en')

print(f'\nJA: {len(docs_ja):,} docs, sample: {docs_ja[0][:80]}...')
print(f'EN: {len(docs_en):,} docs, sample: {docs_en[0][:80]}...')

=== Japanese ===



Wikipedia ja:   0%|          | 0/10000 [00:00<?, ?it/s]


Wikipedia ja:   0%|          | 1/10000 [00:03<9:36:44,  3.46s/it]


Wikipedia ja:  10%|▉         | 997/10000 [00:03<00:22, 395.11it/s]


Wikipedia ja:  16%|█▌        | 1583/10000 [00:06<00:33, 253.67it/s]

...


Wikipedia en:  78%|███████▊  | 7823/10000 [00:07<00:00, 3636.20it/s]


Wikipedia en:  89%|████████▉ | 8892/10000 [00:07<00:00, 4524.49it/s]


Wikipedia en:  99%|█████████▉| 9945/10000 [00:07<00:00, 5458.73it/s]


Wikipedia en: 100%|██████████| 10000/10000 [00:07<00:00, 1342.17it/s]

Collected 10,000 documents (en)

JA: 9,990 docs, sample: アンパサンド（&, ）は、並立助詞「…と…」を意味する記号である。ラテン語で「…と…」を表す接続詞 "et" の合字を起源とする。現代のフォントでも、Trebu...
EN: 10,000 docs, sample: Anarchism is a political philosophy and movement that is skeptical of all justif...


## 2. STSデータセット読み込み

In [3]:
import json
import urllib.request
import io

# JSTS (Japanese STS)
jsts_url = 'https://raw.githubusercontent.com/yahoojapan/JGLUE/v1.1.0/datasets/jsts-v1.1/valid-v1.1.json'
with urllib.request.urlopen(jsts_url) as resp:
    jsts_raw = [json.loads(line) for line in resp.read().decode('utf-8').strip().split('\n')]
print(f'JSTS: {len(jsts_raw)} pairs')

# JSICK (Japanese SICK)
jsick_url = 'https://raw.githubusercontent.com/verypluming/JSICK/b3034994192fae2f41b5937bcf69544e4282fc39/jsick/jsick.tsv'
with urllib.request.urlopen(jsick_url) as resp:
    jsick_df = pd.read_csv(io.StringIO(resp.read().decode('utf-8')), delimiter='\t')
jsick_test = jsick_df[jsick_df['data'] == 'test'].reset_index(drop=True)
print(f'JSICK: {len(jsick_test)} pairs')

# STS-B (English)
stsb_ds = load_dataset('sentence-transformers/stsb', split='test')
print(f'STS-B: {len(stsb_ds)} pairs')

sts_datasets = {
    'JSTS': {
        'sentences1': [d['sentence1'] for d in jsts_raw],
        'sentences2': [d['sentence2'] for d in jsts_raw],
        'scores': [d['label'] for d in jsts_raw],
    },
    'JSICK': {
        'sentences1': jsick_test['sentence_A_Ja'].tolist(),
        'sentences2': jsick_test['sentence_B_Ja'].tolist(),
        'scores': jsick_test['relatedness_score_Ja'].tolist(),
    },
    'STS-B': {
        'sentences1': stsb_ds['sentence1'],
        'sentences2': stsb_ds['sentence2'],
        'scores': stsb_ds['score'],
    },
}

for name, data in sts_datasets.items():
    print(f'{name}: {len(data["sentences1"])} pairs, score range: [{min(data["scores"]):.2f}, {max(data["scores"]):.2f}]')

JSTS: 1457 pairs


JSICK: 4927 pairs


STS-B: 1379 pairs
JSTS: 1457 pairs, score range: [0.00, 5.00]
JSICK: 4927 pairs, score range: [1.00, 5.00]
STS-B: 1379 pairs, score range: [0.00, 1.00]


## 3. ヘルパー関数定義

In [4]:
def clear_gpu():
    """GPU メモリを解放"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def evaluate_embeddings(embeddings: np.ndarray, name: str, n_pairs: int = 10000) -> dict:
    """Embedding品質の評価指標を計算"""
    n = len(embeddings)
    rng = np.random.default_rng(42)
    idx1 = rng.choice(n, n_pairs, replace=True)
    idx2 = rng.choice(n, n_pairs, replace=True)
    mask = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]

    cos_sims = np.sum(embeddings[idx1] * embeddings[idx2], axis=1)

    centered = embeddings - embeddings.mean(axis=0)
    cov = centered.T @ centered / n
    eigenvalues = np.linalg.eigvalsh(cov)[::-1]
    top10_ratio = eigenvalues[:10].sum() / eigenvalues.sum()
    condition_number = eigenvalues[0] / eigenvalues[-1] if eigenvalues[-1] > 1e-10 else float('inf')

    return {
        'model': name,
        'dim': embeddings.shape[1],
        'cos_mean': float(np.mean(cos_sims)),
        'cos_std': float(np.std(cos_sims)),
        'cos_min': float(np.min(cos_sims)),
        'cos_max': float(np.max(cos_sims)),
        'top10_var_ratio': float(top10_ratio),
        'condition_number': float(condition_number),
    }


def compute_sts_metrics(embeddings1: np.ndarray, embeddings2: np.ndarray,
                        gold_scores: list[float]) -> dict:
    """コサイン類似度とゴールドスコアのSpearman/Pearson相関を計算"""
    cos_sims = np.sum(embeddings1 * embeddings2, axis=1)
    gold = np.array(gold_scores)
    sp, sp_pval = spearmanr(cos_sims, gold)
    pe, pe_pval = pearsonr(cos_sims, gold)
    return {'spearman': float(sp), 'pearson': float(pe), 'sp_pval': float(sp_pval), 'pe_pval': float(pe_pval)}


def benchmark_encode(encode_fn, docs: list[str], n_runs: int = 3, warmup: int = 1) -> dict:
    """エンコード関数のベンチマーク"""
    for _ in range(warmup):
        _ = encode_fn(docs[:10])
    times = []
    embeddings = None
    for _ in range(n_runs):
        start = time.time()
        emb = encode_fn(docs)
        elapsed = time.time() - start
        times.append(elapsed)
        if embeddings is None:
            embeddings = emb
    avg_time = np.mean(times)
    return {
        'time': avg_time,
        'std': np.std(times),
        'docs_per_sec': len(docs) / avg_time,
        'ms_per_doc': avg_time / len(docs) * 1000,
        'embeddings': embeddings,
    }


def quick_itq_eval(embeddings: np.ndarray, model_name: str, n_bits: int = 128,
                   n_pairs: int = 10000) -> dict:
    """ITQ-LSHのハッシュ品質を簡易評価"""
    itq = ITQLSH(n_bits=n_bits, n_iterations=50, seed=42)
    itq.fit(embeddings)
    hashes = itq.transform(embeddings)

    rng = np.random.default_rng(42)
    n = len(embeddings)
    idx1 = rng.choice(n, n_pairs, replace=True)
    idx2 = rng.choice(n, n_pairs, replace=True)
    mask = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]

    cos_sims = np.sum(embeddings[idx1] * embeddings[idx2], axis=1)
    ham_dists = np.sum(hashes[idx1] != hashes[idx2], axis=1)
    corr, pval = spearmanr(ham_dists, cos_sims)

    return {
        'model': model_name,
        'n_bits': n_bits,
        'spearman': corr,
        'pvalue': pval,
        'ham_mean': float(np.mean(ham_dists)),
        'ham_std': float(np.std(ham_dists)),
        'itq': itq,
        'hashes': hashes,
    }

print('Helper functions ready.')

Helper functions ready.


## 4. GPU Embedding生成と速度計測

In [5]:
from sentence_transformers import SentenceTransformer

print(f'Loading {MODEL_ID}...')
model = SentenceTransformer(MODEL_ID, device='cuda', trust_remote_code=True)
print(f'Embedding dim: {model.get_sentence_embedding_dimension()}')

# JA (search_document: プレフィックス)
docs_ja_prefixed = [f'search_document: {d}' for d in docs_ja]
start = time.time()
emb_ja = model.encode(docs_ja_prefixed, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
time_ja = time.time() - start
print(f'JA: {emb_ja.shape}, {len(docs_ja)/time_ja:.0f} docs/sec')

# EN
docs_en_prefixed = [f'search_document: {d}' for d in docs_en]
start = time.time()
emb_en = model.encode(docs_en_prefixed, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
time_en = time.time() - start
print(f'EN: {emb_en.shape}, {len(docs_en)/time_en:.0f} docs/sec')

speed_gpu = {
    'ja_docs_sec': len(docs_ja) / time_ja,
    'en_docs_sec': len(docs_en) / time_en,
}
print(f'\nGPU速度: JA {speed_gpu["ja_docs_sec"]:.0f} docs/sec, EN {speed_gpu["en_docs_sec"]:.0f} docs/sec')
print(f'\n--- NB96参考値 (v2-moe GPU) ---')
print(f'  JA: 217 docs/sec, EN: 517 docs/sec')

Loading nomic-ai/nomic-embed-text-v1.5...


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

...

Embedding dim: 768


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

JA: (9990, 768), 228 docs/sec


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

EN: (10000, 768), 767 docs/sec

GPU速度: JA 228 docs/sec, EN 767 docs/sec

--- NB96参考値 (v2-moe GPU) ---
  JA: 217 docs/sec, EN: 517 docs/sec


## 5. Embedding品質分析

In [6]:
result_ja = evaluate_embeddings(emb_ja, MODEL_KEY)
result_en = evaluate_embeddings(emb_en, MODEL_KEY)

print('='*70)
print(f'Embedding品質: {MODEL_KEY}')
print('='*70)
for lang, r in [('JA', result_ja), ('EN', result_en)]:
    print(f'\n--- {lang} ---')
    print(f'  Dim: {r["dim"]}')
    print(f'  Cosine mean: {r["cos_mean"]:.4f} (低いほど等方的)')
    print(f'  Cosine std:  {r["cos_std"]:.4f}')
    print(f'  Cosine range: [{r["cos_min"]:.4f}, {r["cos_max"]:.4f}]')
    print(f'  Top10 var ratio: {r["top10_var_ratio"]:.4f}')
    print(f'  Condition number: {r["condition_number"]:.1f}')

# NB91/NB96参考値
print('\n--- 参考値 (cos_mean) ---')
ref = {
    'nomic_v2_moe': (0.2731, 0.1715),
    'gemma_300m':    (0.360, 0.206),
    'e5_base':       (0.706, 0.594),
    'qwen3_06b':     (0.494, 0.384),
    'bge_m3':        (0.605, 0.459),
}
print(f'{"model":<15} {"JA":>8} {"EN":>8}')
for m, (ja, en) in ref.items():
    print(f'{m:<15} {ja:>8.4f} {en:>8.4f}')
print(f'{MODEL_KEY:<15} {result_ja["cos_mean"]:>8.4f} {result_en["cos_mean"]:>8.4f}')

Embedding品質: nomic_v1_5

--- JA ---
  Dim: 768
  Cosine mean: 0.7052 (低いほど等方的)
  Cosine std:  0.0554
  Cosine range: [0.4294, 0.9507]
  Top10 var ratio: 0.2900
  Condition number: 188486192.0

--- EN ---
  Dim: 768
  Cosine mean: 0.4828 (低いほど等方的)
  Cosine std:  0.0645
  Cosine range: [0.2408, 0.8014]
  Top10 var ratio: 0.1936
  Condition number: 56683520.0

--- 参考値 (cos_mean) ---
model                 JA       EN
nomic_v2_moe      0.2731   0.1715
gemma_300m        0.3600   0.2060
e5_base           0.7060   0.5940
qwen3_06b         0.4940   0.3840
bge_m3            0.6050   0.4590
nomic_v1_5        0.7052   0.4828


## 6. STSベンチマーク評価

In [7]:
# nomicモデルはまだGPU上にある
def encode_nomic(texts):
    return model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)

sts_results = []
print(f'STS Benchmark: {MODEL_KEY}')
print('='*60)

for ds_name, ds_data in sts_datasets.items():
    s1 = [f'search_query: {s}' for s in ds_data['sentences1']]
    s2 = [f'search_query: {s}' for s in ds_data['sentences2']]
    scores = ds_data['scores']

    start = time.time()
    emb1 = encode_nomic(s1)
    emb2 = encode_nomic(s2)
    elapsed = time.time() - start

    metrics = compute_sts_metrics(emb1, emb2, scores)
    metrics['dataset'] = ds_name
    metrics['time_sec'] = elapsed
    sts_results.append(metrics)
    print(f'  {ds_name}: Spearman={metrics["spearman"]:.4f}, Pearson={metrics["pearson"]:.4f} ({elapsed:.1f}s)')

# GPUモデルを解放
del model
clear_gpu()
print('\nGPU model released.')

# NB92/NB96参考値との比較
print('\n--- 参考値 (Spearman) ---')
ref_sts = {
    'nomic_v2_moe':  {'JSTS': 0.7726, 'JSICK': 0.8166, 'STS-B': 0.8343},
    'e5_base':       {'JSTS': 0.8326, 'JSICK': 0.7681, 'STS-B': 0.8488},
    'gemma_300m':    {'JSTS': 0.7779, 'JSICK': 0.7629, 'STS-B': 0.8463},
    'qwen3_06b':     {'JSTS': 0.8452, 'JSICK': 0.7925, 'STS-B': 0.8705},
}
print(f'{"model":<15} {"JSTS":>8} {"JSICK":>8} {"STS-B":>8}')
for m, vals in ref_sts.items():
    print(f'{m:<15} {vals["JSTS"]:>8.4f} {vals["JSICK"]:>8.4f} {vals["STS-B"]:>8.4f}')
nomic_row = {r['dataset']: r['spearman'] for r in sts_results}
print(f'{MODEL_KEY:<15} {nomic_row.get("JSTS", 0):>8.4f} {nomic_row.get("JSICK", 0):>8.4f} {nomic_row.get("STS-B", 0):>8.4f}')

STS Benchmark: nomic_v1_5


  JSTS: Spearman=0.6027, Pearson=0.6053 (0.9s)


  JSICK: Spearman=0.6894, Pearson=0.6872 (2.6s)


  STS-B: Spearman=0.7928, Pearson=0.7865 (0.7s)

GPU model released.

--- 参考値 (Spearman) ---
model               JSTS    JSICK    STS-B
nomic_v2_moe      0.7726   0.8166   0.8343
e5_base           0.8326   0.7681   0.8488
gemma_300m        0.7779   0.7629   0.8463
qwen3_06b         0.8452   0.7925   0.8705
nomic_v1_5        0.6027   0.6894   0.7928


## 7. CPU推論速度比較

In [8]:
# CPU推論（1000件でベンチマーク）
N_CPU_BENCH = 1000
docs_ja_cpu = docs_ja[:N_CPU_BENCH]
docs_en_cpu = docs_en[:N_CPU_BENCH]

print(f'Loading {MODEL_ID} on CPU...')
model_cpu = SentenceTransformer(MODEL_ID, device='cpu', trust_remote_code=True)

def encode_nomic_cpu(docs):
    prefixed = [f'search_document: {d}' for d in docs]
    return model_cpu.encode(prefixed, batch_size=32, normalize_embeddings=True, show_progress_bar=False)

print('\n--- JA ---')
res_cpu_ja = benchmark_encode(encode_nomic_cpu, docs_ja_cpu)
print(f'  {res_cpu_ja["docs_per_sec"]:.1f} docs/sec, {res_cpu_ja["ms_per_doc"]:.1f} ms/doc')

print('--- EN ---')
res_cpu_en = benchmark_encode(encode_nomic_cpu, docs_en_cpu)
print(f'  {res_cpu_en["docs_per_sec"]:.1f} docs/sec, {res_cpu_en["ms_per_doc"]:.1f} ms/doc')

del model_cpu
clear_gpu()

# GPU vs CPU 比較
print('\n' + '='*70)
print('GPU vs CPU 速度比較')
print('='*70)
print(f'{"":<6} {"GPU (docs/s)":>14} {"CPU (docs/s)":>14} {"GPU/CPU ratio":>14}')
print('-' * 50)
for lang, gpu_speed, cpu_res in [
    ('JA', speed_gpu['ja_docs_sec'], res_cpu_ja),
    ('EN', speed_gpu['en_docs_sec'], res_cpu_en),
]:
    ratio = gpu_speed / cpu_res['docs_per_sec']
    print(f'{lang:<6} {gpu_speed:>13.0f} {cpu_res["docs_per_sec"]:>13.1f} {ratio:>13.1f}x')

# 他モデルとのCPU比較
print('\n--- CPU速度 参考値 (1000件) ---')
print(f'  nomic v2-moe: JA 3.8 docs/sec,   EN 10.7 docs/sec  (NB96)')
print(f'  Gemma-300M:   JA ~3  docs/sec,   EN ~8   docs/sec  (NB95)')
print(f'  Qwen3-0.6B:   JA ~1  doc/sec,    EN ~3   docs/sec  (NB95)')
print(f'  E5-base:      JA ~25 docs/sec,   EN ~65  docs/sec  (NB65推定)')

Loading nomic-ai/nomic-embed-text-v1.5 on CPU...


<All keys matched successfully>



--- JA ---


  3.0 docs/sec, 338.5 ms/doc
--- EN ---


  11.3 docs/sec, 88.3 ms/doc

GPU vs CPU 速度比較
         GPU (docs/s)   CPU (docs/s)  GPU/CPU ratio
--------------------------------------------------
JA               228           3.0          77.3x
EN               767          11.3          67.7x

--- CPU速度 参考値 (1000件) ---
  nomic v2-moe: JA 3.8 docs/sec,   EN 10.7 docs/sec  (NB96)
  Gemma-300M:   JA ~3  docs/sec,   EN ~8   docs/sec  (NB95)
  Qwen3-0.6B:   JA ~1  doc/sec,    EN ~3   docs/sec  (NB95)
  E5-base:      JA ~25 docs/sec,   EN ~65  docs/sec  (NB65推定)


## 8. ITQ-LSH適合性 (ビット長最適化)

In [9]:
bit_sizes = [64, 128, 256, 512]
itq_results = []

print('ITQ-LSH ビット長最適化')
print('='*70)

# 128bit結果を保持（保存用）
best_itq = {}
best_hashes = {}

for lang, emb in [('ja', emb_ja), ('en', emb_en)]:
    print(f'\n--- {lang.upper()} (dim={emb.shape[1]}) ---')
    for n_bits in bit_sizes:
        r = quick_itq_eval(emb, MODEL_KEY, n_bits=n_bits)
        r['lang'] = lang
        itq_results.append(r)
        print(f'  ITQ-{n_bits:3d}bit: Spearman={r["spearman"]:.4f}, '
              f'Ham mean={r["ham_mean"]:.1f}, std={r["ham_std"]:.1f}')
        if n_bits == 128:
            best_itq[lang] = r['itq']
            best_hashes[lang] = r['hashes']

# NB91/NB96参考値との比較 (128bit)
print('\n--- 参考値 (ITQ-128bit Spearman) ---')
ref_itq = {
    'nomic_v2_moe':  (-0.6266, -0.6055),
    'e5_base':       (-0.4715, -0.4735),
    'gemma_300m':    (-0.4190, -0.4508),
    'qwen3_06b':     (-0.3919, -0.4176),
}
print(f'{"model":<15} {"JA":>10} {"EN":>10}')
for m, (ja, en) in ref_itq.items():
    print(f'{m:<15} {ja:>10.4f} {en:>10.4f}')
nomic_128 = {r['lang']: r['spearman'] for r in itq_results if r['n_bits'] == 128}
print(f'{MODEL_KEY:<15} {nomic_128.get("ja", 0):>10.4f} {nomic_128.get("en", 0):>10.4f}')

ITQ-LSH ビット長最適化

--- JA (dim=768) ---
ITQ学習開始: samples=9990, dim=768, bits=64
  Centering完了: mean_norm=0.8403
  PCA完了: explained_variance=64.09%


  ITQ iteration 10: quantization_error=0.9137
  ITQ iteration 20: quantization_error=0.9132
  ITQ iteration 30: quantization_error=0.9130
  ITQ iteration 40: quantization_error=0.9129
  ITQ iteration 50: quantization_error=0.9129
ITQ学習完了
  ITQ- 64bit: Spearman=-0.5118, Ham mean=32.0, std=5.6
ITQ学習開始: samples=9990, dim=768, bits=128
  Centering完了: mean_norm=0.8403


  PCA完了: explained_variance=80.33%
  ITQ iteration 10: quantization_error=0.9312


  ITQ iteration 20: quantization_error=0.9307
  ITQ iteration 30: quantization_error=0.9305


  ITQ iteration 40: quantization_error=0.9304
  ITQ iteration 50: quantization_error=0.9304
ITQ学習完了
  ITQ-128bit: Spearman=-0.5321, Ham mean=64.1, std=8.8
ITQ学習開始: samples=9990, dim=768, bits=256
  Centering完了: mean_norm=0.8403


...

  ITQ iteration 10: quantization_error=0.9483


  ITQ iteration 20: quantization_error=0.9480


  ITQ iteration 30: quantization_error=0.9478


  ITQ iteration 40: quantization_error=0.9478


  ITQ iteration 50: quantization_error=0.9477
ITQ学習完了
  ITQ-512bit: Spearman=-0.5796, Ham mean=256.0, std=17.7

--- 参考値 (ITQ-128bit Spearman) ---
model                   JA         EN
nomic_v2_moe       -0.6266    -0.6055
e5_base            -0.4715    -0.4735
gemma_300m         -0.4190    -0.4508
qwen3_06b          -0.3919    -0.4176
nomic_v1_5         -0.5321    -0.5227


## 9. Embedding・ハッシュ保存

In [10]:
# Embedding保存
np.save(DATA_DIR / f'10k_{MODEL_KEY}_ja_embeddings.npy', emb_ja)
np.save(DATA_DIR / f'10k_{MODEL_KEY}_en_embeddings.npy', emb_en)
print(f'Saved: 10k_{MODEL_KEY}_ja_embeddings.npy {emb_ja.shape}')
print(f'Saved: 10k_{MODEL_KEY}_en_embeddings.npy {emb_en.shape}')

# ITQ-128bitハッシュ保存
np.save(DATA_DIR / f'10k_{MODEL_KEY}_ja_hashes_128bits.npy', best_hashes['ja'])
np.save(DATA_DIR / f'10k_{MODEL_KEY}_en_hashes_128bits.npy', best_hashes['en'])
print(f'Saved: 10k_{MODEL_KEY}_ja_hashes_128bits.npy {best_hashes["ja"].shape}')
print(f'Saved: 10k_{MODEL_KEY}_en_hashes_128bits.npy {best_hashes["en"].shape}')

# ITQモデル保存 (JA+EN混合で学習)
emb_mixed = np.vstack([emb_ja, emb_en])
itq_mixed = ITQLSH(n_bits=128, n_iterations=50, seed=42)
itq_mixed.fit(emb_mixed)
itq_path = str(DATA_DIR / f'itq_{MODEL_KEY}_128bits.pkl')
itq_mixed.save(itq_path)
print(f'Saved: itq_{MODEL_KEY}_128bits.pkl (trained on {len(emb_mixed)} mixed samples)')

# 検証
for lang, expected in [('ja', emb_ja), ('en', emb_en)]:
    loaded = np.load(DATA_DIR / f'10k_{MODEL_KEY}_{lang}_embeddings.npy')
    assert np.allclose(loaded, expected), f'{lang} embedding mismatch!'
print('\nVerification passed.')

Saved: 10k_nomic_v1_5_ja_embeddings.npy (9990, 768)
Saved: 10k_nomic_v1_5_en_embeddings.npy (10000, 768)
Saved: 10k_nomic_v1_5_ja_hashes_128bits.npy (9990, 128)
Saved: 10k_nomic_v1_5_en_hashes_128bits.npy (10000, 128)
ITQ学習開始: samples=19990, dim=768, bits=128
  Centering完了: mean_norm=0.7301
  PCA完了: explained_variance=73.56%


  ITQ iteration 10: quantization_error=0.9176
  ITQ iteration 20: quantization_error=0.9170


  ITQ iteration 30: quantization_error=0.9168


  ITQ iteration 40: quantization_error=0.9167
  ITQ iteration 50: quantization_error=0.9166
ITQ学習完了
Saved: itq_nomic_v1_5_128bits.pkl (trained on 19990 mixed samples)



Verification passed.


## 10. 総合サマリー

In [11]:
print('='*90)
print(f'実験97: {MODEL_ID} 総合評価サマリー')
print('='*90)

print(f'\n--- 基本情報 ---')
print(f'  モデル:     {MODEL_ID}')
print(f'  次元:       {emb_ja.shape[1]}')
print(f'  パラメータ: ~100M (Dense)')
print(f'  プレフィックス: search_document: / search_query:')
print(f'  言語:       英語主体')

print(f'\n--- Embedding品質 (cos_mean: 低いほど等方的) ---')
print(f'  JA: {result_ja["cos_mean"]:.4f}  EN: {result_en["cos_mean"]:.4f}')

print(f'\n--- STS Spearman ---')
for r in sts_results:
    print(f'  {r["dataset"]}: {r["spearman"]:.4f}')
avg_sts = np.mean([r['spearman'] for r in sts_results])
print(f'  AVG: {avg_sts:.4f}')

print(f'\n--- GPU推論速度 ---')
print(f'  JA: {speed_gpu["ja_docs_sec"]:.0f} docs/sec')
print(f'  EN: {speed_gpu["en_docs_sec"]:.0f} docs/sec')

print(f'\n--- CPU推論速度 (1000件) ---')
print(f'  JA: {res_cpu_ja["docs_per_sec"]:.1f} docs/sec ({res_cpu_ja["ms_per_doc"]:.1f} ms/doc)')
print(f'  EN: {res_cpu_en["docs_per_sec"]:.1f} docs/sec ({res_cpu_en["ms_per_doc"]:.1f} ms/doc)')

print(f'\n--- ITQ-LSH品質 (Spearman) ---')
for r in itq_results:
    if r['lang'] == 'ja':
        en_r = [x for x in itq_results if x['lang'] == 'en' and x['n_bits'] == r['n_bits']][0]
        print(f'  {r["n_bits"]:3d}bit: JA={r["spearman"]:.4f}  EN={en_r["spearman"]:.4f}')

print(f'\n--- 保存ファイル ---')
for f in sorted(DATA_DIR.glob(f'*{MODEL_KEY}*')):
    print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')

# v2-moeとの直接比較テーブル
print('\n' + '='*90)
print('v1.5 vs v2-moe 直接比較')
print('='*90)
v2 = {
    'cos_mean_ja': 0.2731, 'cos_mean_en': 0.1715,
    'sts_jsts': 0.7726, 'sts_jsick': 0.8166, 'sts_stsb': 0.8343,
    'gpu_ja': 217, 'gpu_en': 517,
    'cpu_ja': 3.8, 'cpu_en': 10.7,
    'itq128_ja': -0.6266, 'itq128_en': -0.6055,
}
v1_sts = {r['dataset']: r['spearman'] for r in sts_results}
v1_itq = {r['lang']: r['spearman'] for r in itq_results if r['n_bits'] == 128}

rows = [
    ('cos_mean JA', result_ja['cos_mean'], v2['cos_mean_ja']),
    ('cos_mean EN', result_en['cos_mean'], v2['cos_mean_en']),
    ('JSTS Spearman', v1_sts.get('JSTS', 0), v2['sts_jsts']),
    ('JSICK Spearman', v1_sts.get('JSICK', 0), v2['sts_jsick']),
    ('STS-B Spearman', v1_sts.get('STS-B', 0), v2['sts_stsb']),
    ('GPU JA (docs/s)', speed_gpu['ja_docs_sec'], v2['gpu_ja']),
    ('GPU EN (docs/s)', speed_gpu['en_docs_sec'], v2['gpu_en']),
    ('CPU JA (docs/s)', res_cpu_ja['docs_per_sec'], v2['cpu_ja']),
    ('CPU EN (docs/s)', res_cpu_en['docs_per_sec'], v2['cpu_en']),
    ('ITQ-128 JA', v1_itq.get('ja', 0), v2['itq128_ja']),
    ('ITQ-128 EN', v1_itq.get('en', 0), v2['itq128_en']),
]
print(f'{"metric":<20} {"v1.5":>12} {"v2-moe":>12} {"diff":>12}')
print('-' * 58)
for name, v1_val, v2_val in rows:
    diff = v1_val - v2_val
    print(f'{name:<20} {v1_val:>12.4f} {v2_val:>12.4f} {diff:>+12.4f}')

print('\n' + '='*90)

実験97: nomic-ai/nomic-embed-text-v1.5 総合評価サマリー

--- 基本情報 ---
  モデル:     nomic-ai/nomic-embed-text-v1.5
  次元:       768
  パラメータ: ~100M (Dense)
  プレフィックス: search_document: / search_query:
  言語:       英語主体

--- Embedding品質 (cos_mean: 低いほど等方的) ---
  JA: 0.7052  EN: 0.4828

--- STS Spearman ---
  JSTS: 0.6027
  JSICK: 0.6894
  STS-B: 0.7928
  AVG: 0.6950

--- GPU推論速度 ---
  JA: 228 docs/sec
  EN: 767 docs/sec

--- CPU推論速度 (1000件) ---
  JA: 3.0 docs/sec (338.5 ms/doc)
  EN: 11.3 docs/sec (88.3 ms/doc)

--- ITQ-LSH品質 (Spearman) ---
   64bit: JA=-0.5118  EN=-0.4734
  128bit: JA=-0.5321  EN=-0.5227
  256bit: JA=-0.5619  EN=-0.5389
  512bit: JA=-0.5946  EN=-0.5796

--- 保存ファイル ---
  10k_nomic_v1_5_en_embeddings.npy (30000 KB)
  10k_nomic_v1_5_en_hashes_128bits.npy (1250 KB)
  10k_nomic_v1_5_ja_embeddings.npy (29970 KB)
  10k_nomic_v1_5_ja_hashes_128bits.npy (1249 KB)
  itq_nomic_v1_5_128bits.pkl (451 KB)

v1.5 vs v2-moe 直接比較
metric                       v1.5       v2-moe         diff
--------------

## 評価と考察

### 1. v1.5 vs v2-moe 直接比較

| 指標 | v1.5 | v2-moe | 差 |
|------|------|--------|-----|
| cos_mean JA | 0.705 | **0.273** | v2が大幅に等方的 |
| cos_mean EN | 0.483 | **0.172** | 同上 |
| JSTS Spearman | 0.603 | **0.773** | v2が**+17pp** |
| JSICK Spearman | 0.689 | **0.817** | v2が**+13pp** |
| STS-B Spearman | 0.793 | **0.834** | v2が+4pp |
| GPU EN (docs/s) | **767** | 517 | v1.5が**+48%**高速 |
| GPU JA (docs/s) | 228 | 217 | ほぼ同等 |
| CPU EN (docs/s) | 11.3 | 10.7 | ほぼ同等 |
| CPU JA (docs/s) | 3.0 | 3.8 | v2がやや速い |
| ITQ-128 JA | -0.532 | **-0.627** | v2が**+18%**良い |
| ITQ-128 EN | -0.523 | **-0.606** | v2が**+16%**良い |

### 2. 日本語は予想通り厳しい

- cos_mean JA=0.705はE5-base(0.706)とほぼ同じで、日本語では等方性が大幅に劣化
- JSTS Spearman=0.603は全モデル中最低水準（E5-base 0.833, v2-moe 0.773）
- **v1.5の日本語利用は非推奨**。英語主体モデルの限界が明確

### 3. CPU推論速度: v2-moeと同等 — 期待外れ

- CPU EN: v1.5=11.3, v2-moe=10.7 → **ほぼ差なし**
- CPU JA: v1.5=3.0, v2-moe=3.8 → v2-moeの方がやや速い
- 100M Dense vs 305M active MoEで速度差がほとんどない原因:
  - v1.5は最大8192トークン対応のため、Attention層が深い可能性
  - MoEはtop-2ルーティングで実効的な計算量が抑制されている
- **CPU速度でv1.5を選ぶ理由はない**

### 4. GPU推論速度: EN方向ではv1.5が優位

- GPU EN: v1.5=767 vs v2-moe=517 → **v1.5が+48%高速**
- これはDenseアーキテクチャの利点（GPU並列効率が高い）
- ただし英語専用かつ品質面でv2-moeに劣るため、速度だけでは選択理由にならない

### 5. ITQ-LSH品質: v2-moeに及ばないが、E5-baseは上回る

- v1.5 128bit: JA=-0.532, EN=-0.523
- E5-base 128bit: JA=-0.472, EN=-0.474
- v2-moe 128bit: JA=-0.627, EN=-0.606
- v1.5はE5-baseよりITQ相性が良い（+12-10%）が、v2-moeには明確に劣る

### 6. 総合評価

| 観点 | v1.5 | v2-moe |
|------|------|--------|
| 等方性 | ★☆☆ (EN中位, JA低) | ★★★ (全モデル最良) |
| ITQ-LSH相性 | ★★☆ (E5超え) | ★★★ (テキスト最高) |
| STS品質 | ★☆☆ (JA不可) | ★★☆ (中上位) |
| GPU速度 | ★★★ (EN最速級) | ★★☆ (中位) |
| CPU速度 | ★☆☆ (v2と同等) | ★☆☆ (非実用的) |
| 日本語対応 | ✗ | ✓ |

**結論**: v1.5は英語GPUオンリーの用途ではGPU速度が魅力だが、日本語が使えず、CPU速度もv2-moeと同等であるため、**本PoCの用途ではv2-moeが全面的に優位**。v1.5を採用する積極的な理由は見当たらない。NB118のVoronoi実験はv2-moeで進める。

### 7. 未検証事項（将来の検証候補）

- **長文入力での品質比較**: v1.5は最大8192トークン、v2-moeは最大512トークン。本実験ではMAX_CHARS=500（短文）でのみ評価しており、長文での品質差は未検証。チャンクせずに長文をそのまま埋め込む用途では、v1.5の8192トークン対応が優位になる可能性がある
- **長文でのSTS・ITQ品質変化**: 512トークンを超える入力に対してv2-moeが切り捨てる影響がどの程度検索品質に影響するか
- **チャンキング戦略との組み合わせ**: NB101-105の推奨チャンクサイズ（E5-base: 256トークン/64オーバーラップ）であればv2-moeの512制限は問題ないが、より大きなチャンクサイズでの比較は未実施
- **Matryoshka次元削減での比較**: v1.5は64次元まで、v2-moeは256次元まで縮小可能。低次元での品質・速度トレードオフは未検証